In [ ]:
"""
Lookup Metadata Join — additive enrichment step
================================================
Input:  harmonised output from 00_harmonisation.ipynb
        + reference lookups (gene_lookup, union_lookup_entity)
Output: harmonised structure + 4 metadata columns that exist ONLY after
        lookup-building logic (verified via provenance check 2026-07-19):
          - biotype, hgnc_status          (from gene_lookup, join on ensg_id)
          - canonical_name, gap_class     (from union_lookup_entity, join on model_id)
          
Why these matter:
  - biotype / hgnc_status : exclude withdrawn + non-coding genes before scoring
  - gap_class             : flag ambiguous identity links before any downstream join
  - canonical_name        : normalised display name for dedup / reporting

Author: Chaithali
Harmonisation input authored by: Triveni
Date: 2026-07-19
"""

import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "scripts")))

import pandas as pd
from pathlib import Path

REF_DIR      = Path(os.path.abspath(os.path.join(os.getcwd(), "..", "..", "reference")))
PIPELINE_DIR = Path(os.getcwd())
OUT_DIR      = PIPELINE_DIR / "outputs"
OUT_DIR.mkdir(exist_ok=True)

print("REF_DIR:", REF_DIR, "| exists:", REF_DIR.is_dir())
print("OUT_DIR:", OUT_DIR)

In [ ]:
# ---------------------------------------------------------------------------
# Cell 2 — Load harmonised output as read-only input
# ---------------------------------------------------------------------------
# harmonised.parquet contains list-typed columns (rrids, profile_ids, etc.)
# written by Triveni's harmonisation notebook. pd.read_parquet cannot convert
# list<string>[pyarrow] back to a pandas dtype — strip the stale pandas metadata
# and cast list columns to JSON strings so the rest of the pipeline can read it.

import pyarrow.parquet as pq
import pyarrow as pa
import json

def read_parquet_with_lists(path):
    tbl = pq.read_table(path)
    new_meta = {k: v for k, v in tbl.schema.metadata.items() if k != b"pandas"}
    tbl = tbl.replace_schema_metadata(new_meta)
    for field in tbl.schema:
        if pa.types.is_list(field.type):
            arr = tbl.column(field.name)
            as_str = pa.array(
                [json.dumps(x.as_py()) if x.is_valid else None for x in arr],
                type=pa.string()
            )
            tbl = tbl.set_column(tbl.schema.get_field_index(field.name), field.name, as_str)
    return tbl.to_pandas()

HARMONISED_PATH = OUT_DIR / "harmonised.parquet"

if not HARMONISED_PATH.exists():
    raise FileNotFoundError(
        f"Harmonised output not found at {HARMONISED_PATH}.\n"
        "Run 00_harmonisation.ipynb first and save cell_line_connection:\n"
        "  cell_line_connection.to_parquet('outputs/harmonised.parquet', index=False)"
    )

harmonised = read_parquet_with_lists(HARMONISED_PATH)
print(f"Harmonised input: {harmonised.shape}")
print(f"Columns: {harmonised.columns.tolist()}")
print(f"Keys present: model_id={'model_id' in harmonised.columns}, "
      f"ensg_id={'ensg_id' in harmonised.columns}")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 3 — Load reference lookups, inspect actual columns
# ---------------------------------------------------------------------------
# Provenance-verified column names (checked 2026-07-19):
#   gene_lookup        : ensg_id, biotype, hgnc_status  (no canonical_name here)
#   union_lookup_entity: primary_cvcl, model_id_set (list), gap_class, canonical_name
# ---------------------------------------------------------------------------

gene_lookup   = pd.read_parquet(REF_DIR / "gene_lookup.parquet")
union_lookup  = pd.read_parquet(REF_DIR / "union_lookup_entity.parquet")

print("gene_lookup columns:  ", gene_lookup.columns.tolist())
print("gene_lookup shape:    ", gene_lookup.shape)
print()
print("union_lookup columns: ", union_lookup.columns.tolist())
print("union_lookup shape:   ", union_lookup.shape)
print()
print("gap_class value counts:")
print(union_lookup["gap_class"].value_counts())
print()
print("model_id_set sample:", union_lookup["model_id_set"].head(3).tolist())

In [ ]:
# ---------------------------------------------------------------------------
# Cell 4 — Build join tables from the reference lookups
# ---------------------------------------------------------------------------
# Keys are lowercased to match the harmonised output (harmonisation lowercases
# all model_id and ensg_id values).

# Gene-level: biotype + hgnc_status from gene_lookup (join key: ensg_id)
gene_meta = gene_lookup[["ensg_id", "biotype", "hgnc_status"]].copy()
gene_meta["ensg_id"] = gene_meta["ensg_id"].astype("string").str.split(".").str[0].str.lower()

print(f"gene_meta: {gene_meta.shape}  — ensg_id sample: {gene_meta['ensg_id'].head(3).tolist()}")

# Cell-line-level: canonical_name + gap_class from union_lookup_entity
# model_id_set is a list column — explode to one model_id per row, then deduplicate
cell_meta = (
    union_lookup[["model_id_set", "canonical_name", "gap_class"]]
    .explode("model_id_set")
    .rename(columns={"model_id_set": "model_id"})
    .dropna(subset=["model_id"])
    .drop_duplicates(subset=["model_id"])
    .copy()
)
cell_meta["model_id"] = cell_meta["model_id"].astype("string").str.lower()

print(f"cell_meta: {cell_meta.shape}  — model_id sample: {cell_meta['model_id'].head(3).tolist()}")
print(f"Any duplicate model_id after dedup: {cell_meta['model_id'].duplicated().sum()}")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 5 — Additive left joins (harmonised is always the left side)
# ---------------------------------------------------------------------------
# Keys lowercased to match harmonisation output.

enriched = harmonised.copy()

# --- Gene-level join (only if ensg_id exists in harmonised) ---
if "ensg_id" in enriched.columns:
    enriched["ensg_id"] = enriched["ensg_id"].astype("string").str.split(".").str[0].str.lower()
    before = len(enriched)
    enriched = enriched.merge(gene_meta, on="ensg_id", how="left")
    assert len(enriched) == before, f"Gene join fanned out: {before} -> {len(enriched)} rows"
    print(f"Gene join OK — rows unchanged: {len(enriched)}")
else:
    print("WARNING: ensg_id not in harmonised — gene-level columns (biotype, hgnc_status) skipped.")
    enriched["biotype"]     = pd.NA
    enriched["hgnc_status"] = pd.NA

# --- Cell-line join (canonical_name + gap_class on model_id) ---
enriched["model_id"] = enriched["model_id"].astype("string").str.lower()
before = len(enriched)
enriched = enriched.merge(cell_meta, on="model_id", how="left")
assert len(enriched) == before, f"Cell-line join fanned out: {before} -> {len(enriched)} rows"
print(f"Cell-line join OK — rows unchanged: {len(enriched)}")

print(f"\nBefore enrichment: {harmonised.shape}")
print(f"After enrichment:  {enriched.shape}")
print(f"New columns added: {[c for c in enriched.columns if c not in harmonised.columns]}")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 6 — Match rate audit
# ---------------------------------------------------------------------------
# Low match rates = key mismatch (case, version suffix, wrong grain),
# not genuinely missing metadata. Investigate before accepting a low number.
#   biotype / hgnc_status : expect near 100% for valid protein-coding ENSGs
#   canonical_name / gap_class : expect near 100% for ACHs in the DepMap roster

print("=== Match rates ===")
for col in ["biotype", "hgnc_status", "canonical_name", "gap_class"]:
    if col in enriched.columns:
        matched = enriched[col].notna().sum()
        pct = 100 * matched / len(enriched)
        flag = "" if pct >= 90 else "  <-- INVESTIGATE KEY MISMATCH"
        print(f"  {col:<18}: {matched:>6}/{len(enriched)} rows matched ({pct:5.1f}%){flag}")
    else:
        print(f"  {col:<18}: column not present")

print()
# Show unmatched model_ids for gap_class (if any)
if "gap_class" in enriched.columns:
    unmatched = enriched.loc[enriched["gap_class"].isna(), "model_id"].dropna().unique()
    if len(unmatched):
        print(f"Unmatched model_ids (gap_class): {len(unmatched)} — sample: {unmatched[:5]}")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 7 — Write enriched output (never overwrites harmonised input)
# ---------------------------------------------------------------------------

OUT_PATH = OUT_DIR / "harmonised_enriched.parquet"
enriched.to_parquet(OUT_PATH, index=False)

print(f"Written : {OUT_PATH}")
print(f"Shape   : {enriched.shape}")
print(f"Columns : {enriched.columns.tolist()}")
print(f"\nHarmonisation input untouched at: {HARMONISED_PATH}")